In [1]:
# declare libraries
import json
import geojson
import os
import pathlib
import time
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely

from zipfile import ZipFile
import rioxarray as rxr
import xarray as xr
import rasterio
from rasterio.plot import show

import pprint as pp

import matplotlib.pyplot as plt

import requests
from requests.auth import HTTPBasicAuth
from planet import Session, DataClient, OrdersClient, Auth, Planet

import os
import sys

sys.path.append("../utils")

import config
import planet_api

import pyproj

# Point this to where your actual proj.db is (inside your conda env)
# pyproj.datadir.set_data_dir("/Users/rachelfswick/.conda/envs/wildfire_prep/share/proj") 

from pyproj import CRS

# declare misc vars
crs = "EPSG:4326"
#crs = "EPSG:3310"
county_filepath = "/capstone/wildfire_prep/data/ca_counties/CA_Counties.shp"
# sb_bbox = [-125, 34.25, -119.0, 38.0]

data_api_url = "https://api.planet.com/data/v1"
orders_api_url = 'https://api.planet.com/compute/ops/orders/v2' 

# load CA county shps
counties = (
   gpd
   .read_file(county_filepath)
   .to_crs(crs)
  )

# isolate sb shp
sb_county = (
    counties
    .loc[counties["NAME"] == "Santa Barbara"]
    .explode(ignore_index=True)
    .iloc[[0]]
    )

# import 2019 buffer geometries
with open('/capstone/wildfire_prep/data/buffer_geometries/buffer_geometries_2019.geojson', 'r') as file:
    buffer_2019 = dict(geojson.load(file))["features"]


# planet_api = config.planet_api

In [2]:
# test buffer geojson by printing it
pp.pprint(
    buffer_2019[0]["geometry"]["coordinates"][0]
    )

[[-2813.178692, -371821.557405],
 [-2813.178692, -371905.216569],
 [-2862.441447, -371905.216569],
 [-2862.034816, -371898.471],
 [-2884.060993, -371840.700116],
 [-2874.776725, -371821.557405],
 [-2813.178692, -371821.557405]]


In [3]:
# make subset
buffer_subset = buffer_2019[:5]

pp.pprint(buffer_subset)

[{"geometry": {"coordinates": [[[-2813.178692, -371821.557405], [-2813.178692, -371905.216569], [-2862.441447, -371905.216569], [-2862.034816, -371898.471], [-2884.060993, -371840.700116], [-2874.776725, -371821.557405], [-2813.178692, -371821.557405]]], "type": "Polygon"}, "properties": {"Date": "2019-06-05T00:00:00", "a_removebr": null, "accessegre": "yes", "address__1": null, "address__2": "Santa Barbara", "address_ad": "CA", "address_co": "US", "address_fu": "3462 Brinkerhoff Rd Santa Ynez Santa Barbara CA 93460 US", "address_lo": "Santa Ynez", "address_po": "93460", "address_su": "3462", "address_th": "Brinkerhoff Rd", "addressvis": "Yes - Without Reflective", "apn": "141-010-040", "appenddate": "2019-06-05T00:00:00", "assigned_t": null, "b_removele": null, "battalion": "2", "c_removede": null, "calculated": null, "calfireuni": "SBC", "can_engine": "yes", "citationda": null, "citationnu": null, "clearreins": null, "community": "Woodstock", "community_": null, "core_with_": null, "

### Create AOI geojson

##### This function creates individual Feature collections of set number of polygons for any specified window, or a set number of vertices within the buffer geojsons. They are (hopefully) in the format that the API is looking for. 

This will be useful for our stress testing.

In [46]:
def make_aoi_geojson(buffers_set, max_verts = float("inf"), begin = 0, end = -1):

    # init geojson structure for master list
    geojson = {
        "type": "FeatureCollection", 
        "features": []
    }

    vert_count = 0

    # for every buffer within a window of the buffer collection
    for buffer in buffers_set[begin:end]:

        curr_buffer = (
                        buffer
                        ["geometry"]["coordinates"] # grab the coords specifically
                        [0] # unlists one stage, for formatting
                    )

        feature = {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    curr_buffer
                ]
            }
        }

        # append to master list
        geojson["features"].append(feature)

        vert_count = vert_count + len(curr_buffer)
        if vert_count > max_verts:
            vert_count = vert_count - len(curr_buffer)
            geojson["features"].pop(-1)
            print(f"Max vert limit reached. Total vertices in this AOI filter: {vert_count}")
            return geojson

    print(f"Total vertices in this AOI filter: {vert_count}")
        

    return geojson

Testing it...

In [ ]:
# make test cases for stress test
poly_1 = make_aoi_geojson(buffer_2019, end = 1)
poly_10 = make_aoi_geojson(buffer_2019, end = 10)
poly_25 = make_aoi_geojson(buffer_2019, end = 25)
poly_30 = make_aoi_geojson(buffer_2019, end = 30)
poly_40 = make_aoi_geojson(buffer_2019, end = 40)

vert_10 = make_aoi_geojson(buffer_2019, max_verts = 10)
vert_25 = make_aoi_geojson(buffer_2019, max_verts = 25)
vert_60 = make_aoi_geojson(buffer_2019, max_verts = 60)
vert_100 = make_aoi_geojson(buffer_2019, max_verts = 100)
vert_150 = make_aoi_geojson(buffer_2019, max_verts = 150)
vert_200 = make_aoi_geojson(buffer_2019, max_verts = 200)
vert_300 = make_aoi_geojson(buffer_2019, max_verts = 300)
vert_400 = make_aoi_geojson(buffer_2019, max_verts = 400)

print(f"Number of orders if we use poly-1: {len(buffer_2019)/1}")
print(f"Number of orders if we use poly-10: {len(buffer_2019)/10}")
print(f"Number of orders if we use poly-25: {len(buffer_2019)/25}")
print(f"Number of orders if we use poly-30: {len(buffer_2019)/30}")
print(f"Number of orders if we use poly-40: {len(buffer_2019)/40}")

Total vertices in this AOI filter: 7
Total vertices in this AOI filter: 59
Total vertices in this AOI filter: 140
Total vertices in this AOI filter: 168
Total vertices in this AOI filter: 227
Number of orders if we use poly-1: 17211.0
Number of orders if we use poly-10: 1721.1
Number of orders if we use poly-25: 688.44
Number of orders if we use poly-30: 573.7
Number of orders if we use poly-40: 430.275


##### This one creates a list of FeatureCollections, such that they can be iterated through. Will be useful for when we start making all our orders. 

In [6]:
def make_aoi_geojson_collection(buffers_set, groups_of = 1):

    # buffers_set is one of the buffers geojsons that Lei generated
    # groups_of is the number of polygons per feature collection

    # init
    geojson_collection = [] # master list for all feature collections
    begin = 0 # starting index of window
    end = groups_of # ending index of window

    # loop will repeat until the top of the window is greater than the actual number of polygons in the buffer set
    while end < len(buffers_set) + groups_of:
        print(f"step: begin = {begin}, end = {end}")

        # and when that happens, force the top of the window to equal the total number of polygons
        if end > len(buffers_set) + groups_of:
            end = len(buffers_set)

        # declare feature collection structure
        geojson = {
            "type": "FeatureCollection", 
            "features": []
        }

        # for every buffer within a window of the buffer collection
        for buffer in buffers_set[begin:end]:

            feature = {
                "type": "Feature",
                "properties": {},
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [
                        (
                            buffer
                            ["geometry"]["coordinates"] # grab the coords specifically
                            [0] # unlists one stage, for formatting
                        )
                    ]
                }
            }

            # append to current feature collection
            geojson["features"].append(feature)
    
        # update window for next set of polygons
        begin = begin + groups_of
        end = end + groups_of

        # append current feature collection to master collection
        geojson_collection.append(geojson)
        
        

    return geojson_collection

In [7]:
test = make_aoi_geojson_collection(buffer_2019[:43], 13)

test[3]

step: begin = 0, end = 13
step: begin = 13, end = 26
step: begin = 26, end = 39
step: begin = 39, end = 52


{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'properties': {},
   'geometry': {'type': 'Polygon',
    'coordinates': [[[-6490.801527, -368092.167337],
      [-6402.428372, -368092.167337],
      [-6402.428372, -368172.647299],
      [-6486.743253, -368172.647299],
      [-6490.801527, -368160.127222],
      [-6490.801527, -368092.167337]]]}},
  {'type': 'Feature',
   'properties': {},
   'geometry': {'type': 'Polygon',
    'coordinates': [[[-5474.074748, -368273.922844],
      [-5474.074748, -368381.694845],
      [-5581.767386, -368381.694845],
      [-5581.767386, -368363.765442],
      [-5549.238058, -368321.159236],
      [-5506.345072, -368273.922844],
      [-5474.074748, -368273.922844]]]}},
  {'type': 'Feature',
   'properties': {},
   'geometry': {'type': 'Polygon',
    'coordinates': [[[-5745.257336, -369376.018966],
      [-5854.365732, -369376.018966],
      [-5854.365732, -369269.587855],
      [-5760.290116, -369269.587855],
      [-5745.257336, -369

In [15]:
len(buffer_2019[12]["geometry"]["coordinates"][0])

5

In [45]:
def make_aoi_geojson(buffers_set, max_verts = float("inf"), begin = 0, end = -1):

    # init geojson structure for master list
    geojson = {
        "type": "FeatureCollection", 
        "features": []
    }

    vert_count = 0

    # for every buffer within a window of the buffer collection
    for buffer in buffers_set[begin:end]:

        curr_buffer = (
                        buffer
                        ["geometry"]["coordinates"] # grab the coords specifically
                        [0] # unlists one stage, for formatting
                    )

        feature = {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    curr_buffer
                ]
            }
        }

        # append to master list
        geojson["features"].append(feature)

        vert_count = vert_count + len(curr_buffer)
        if vert_count > max_verts:
            vert_count = vert_count - len(curr_buffer)
            geojson["features"].pop(-1)
            print(f"Max vert limit reached. Total vertices in this AOI filter: {vert_count}")
            return geojson

    print(f"Total vertices in this AOI filter: {vert_count}")
        

    return geojson

# DEPRICATION ZONE

In [ ]:
# def make_aoi_geojson(buffers_set, begin = 0, end = -1):

#     # init geojson structure for master list
#     geojson = {
#         "type": "FeatureCollection", 
#         "features": []
#     }

#     # for every buffer within a window of the buffer collection
#     for buffer in buffers_set[begin:end]:

#         feature = {
#             "type": "Feature",
#             "properties": {},
#             "geometry": {
#                 "type": "Polygon",
#                 "coordinates": [
#                     (
#                         buffer
#                         ["geometry"]["coordinates"] # grab the coords specifically
#                         [0] # unlists one stage, for formatting
#                     )
#                 ]
#             }
#         }

#         # append to master list
#         geojson["features"].append(feature)
        

#     return geojson

In [ ]:
# # make test cases for stress test
# poly_1 = make_aoi_geojson(buffer_2019, end = 1)
# poly_10 = make_aoi_geojson(buffer_2019, end = 10)
# poly_25 = make_aoi_geojson(buffer_2019, end = 25)
# poly_30 = make_aoi_geojson(buffer_2019, end = 30)
# poly_40 = make_aoi_geojson(buffer_2019, end = 40)

# print(f"Number of orders if we use poly-1: {len(buffer_2019)/1}")
# print(f"Number of orders if we use poly-10: {len(buffer_2019)/10}")
# print(f"Number of orders if we use poly-25: {len(buffer_2019)/25}")
# print(f"Number of orders if we use poly-30: {len(buffer_2019)/30}")
# print(f"Number of orders if we use poly-40: {len(buffer_2019)/40}")